# Заметки \#5

## $B$-сплайны

$B$-сплайны (Basis splines) — семейство кусочно-полиномиальных функций, используемых для аппроксимации и представления кривых и поверхностей. $B$-сплайны задаются с помощью:
- множества опорных (контрольных) точек $P_i$,
- узлового вектора (knot vector) $t = [t_0, t_1, \dots, t_{m}]$,
- степени сплайна $p$ (обычно $p=1$ — линейный, $p=2$ — квадратичный, $p=3$ — кубический и т.д.).

Размеры связаны так:
- число опорных точек: $n+1$ (индексы $0\ldots n$),
- длина узлового вектора: $m+1 = n + p + 2$.

Параметр $u$ изменяется в диапазоне $[t_p, t_{n+1}]$.


Узловой вектор $t$ — неубывающая последовательность чисел:
$$
t_0 \leqslant t_1 \leqslant \dots \leqslant t_m.
$$
    
Типичные варианты:
- **Равномерный** (uniform): узлы равномерно распределены.
- **Неравномерный** (non-uniform): произвольные узлы.
- **Клэменный** (clamped): первые и последние узлы повторяются $p+1$ раз — дает кривой, проходящую через первые и последние опорные точки.

Пример клэмпированного узлового вектора для кубического сплайна $p=3$, $n+1=6$:
$$
t = [0,0,0,0,1,2,3,3,3,3] \quad (m+1 = 10).
$$


## Определение базисных функций: алгоритм Кокса — де Бура

Базисные функции $N_{i,p}(u)$ определяются рекурсивно:

Для $p = 0$:
$$
N_{i,0}(u) =
\begin{cases}
1, & t_i \le u < t_{i+1},\\
0, & \text{иначе}.
\end{cases}
$$

Для $p \geqslant 1$:
$$
N_{i,p}(u) =
\frac{u - t_i}{t_{i+p} - t_i} N_{i,p-1}(u)
+ \frac{t_{i+p+1} - u}{t_{i+p+1} - t_{i+1}} N_{i+1,p-1}(u),
$$
при условии, что деления на ноль трактуются как ноль (если знаменатель = 0, соответствующая дробь считается 0).


Рассмотрим построение для $p = 3$:

У нас по определению:
$$
N_{i,3}(u) =
\frac{u - t_i}{t_{i+3} - t_i} N_{i,2}(u)
+ \frac{t_{i+4} - u}{t_{i+4} - t_{i+1}} N_{i+1,2}(u).
$$

А $N_{i,2}(u)$ и $N_{i+1,2}(u)$ в свою очередь через квадратичные функции:

$$
N_{i,2}(u) =
\frac{u - t_i}{t_{i+2} - t_i} N_{i,1}(u)
+ \frac{t_{i+3} - u}{t_{i+3} - t_{i+1}} N_{i+1,1}(u),
$$

$$
N_{i+1,2}(u) =
\frac{u - t_{i+1}}{t_{i+3} - t_{i+1}} N_{i+1,1}(u)
+ \frac{t_{i+4} - u}{t_{i+4} - t_{i+2}} N_{i+2,1}(u).
$$

А линейные $N_{i,1}(u)$ уже через нулевые:

$$
N_{i,1}(u) =
\frac{u - t_i}{t_{i+1} - t_i} N_{i,0}(u)
+ \frac{t_{i+2} - u}{t_{i+2} - t_{i+1}} N_{i+1,0}(u).
$$

## $B$-сплайн кривая

Кривая $C(u)$ степени $p$ задаётся как линейная комбинация базисных функций и опорных точек:
$$
C(u) = \sum_{i=0}^{n} N_{i,p}(u) \, P_i, \quad u \in [t_p, t_{n+1}].
$$

Если используются веса $w_i$ (рациональные $B$-сплайны — NURBS (Non-Uniform Rational $B$-Splines
(неравномерные рациональные $B$-сплайны))), то:
$$
C(u) = \frac{\sum_{i=0}^n N_{i,p}(u) w_i P_i}{\sum_{i=0}^n N_{i,p}(u) w_i}.
$$

## Алгоритм де Бура

Алгоритм де Бура — численно устойчивый метод вычисления $C(u)$, эквивалентный применению базисов, но гораздо эффективнее и стабильнее.

Псевдокод (идея):
1. Найти индекс `k` такой, что $t_k \le u < t_{k+1}$.
2. Создать массив `d[j] = P_{k-p+j}`, для $j = 0..p$.
3. Для `r = 1..p`:
   - Для `j = p..r` (обратно):  
     $ \alpha = \frac{u - t_{k-p+j}}{t_{k+j-r+1} - t_{k-p+j}}$
     $ d[j] = (1-\alpha) d[j-1] + \alpha d[j] $
4. Тогда $C(u) = d[p]$.

## Свойства $B$-сплайнов

- Локальность: $N_{i,p}(u)$ ненулева только на отрезке $[t_i, t_{i+p+1})$.
- Ненегативность: $N_{i,p}(u) \ge 0$.
- Линейная комбинация даёт выпуклую комбинацию: кривая находится в выпуклом объединении соответствующих контрольных точек.
- Непрерывность: при общем положении узлов кривая имеет $C^{p-1}$-непрерывность; кратные узлы снижают гладкость.
- Параметризация: выбор узлового вектора влияет на форму и распределение точек по $u$.
